In [134]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [135]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 


In [173]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 2, 1, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 2, 23, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="SEC",
	asset_class="CREDITS",
)
df

MERGING SLICES...: 100%|██████████| 16/16 [00:00<00:00, 325.45it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,1984835184000000101,,NEWT,TRAD,2026-02-02 00:00:00+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZLX6S2LRQBS,NA/CDS Corp SN Jr,"THE CHILDREN'S PLACE, INC."
1,1984835184000000201,,NEWT,TRAD,2026-02-02 00:00:00+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZLX6S2LRQBS,NA/CDS Corp SN Jr,"THE CHILDREN'S PLACE, INC."
2,1931337777000000401,1451916031,MODI,TRAD,2026-02-02 01:02:48+00:00,False,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,
3,1931337775000000201,1451916257,CORR,,2026-02-02 01:02:48+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,
4,1931337783000001001,1451915842,CORR,,2026-02-02 01:02:49+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24847,2146189130000000101,1451916257,CORR,,2026-02-24 04:49:03+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,
24848,2146189136000000701,1451915842,CORR,,2026-02-24 04:49:06+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,
24849,2146189140000001101,1451916031,CORR,,2026-02-24 04:49:09+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,
24850,2146189131000000201,1451916717,CORR,,2026-02-24 04:49:10+00:00,None,CR,None,N,None,...,None,NaN,NaN,None,NaN,None,None,QZM6K4SR2Q9S,NA/Cr Sw Bskt Oth Corp Au,


In [174]:
# ["Identifier_UPI"]
# cols = [
#     "Identifier_UPI",
#     "Derived_UnderlierName"
# ]
u = pd.read_csv(r'C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Option-Debt_Option.csv')
# print([x for x in list(u.columns) if "isin" in str(x).lower()])
upis = u[u["Attributes_UnderlyingInstrumentISIN"].isin(pd.read_csv(r"C:\Users\chris\Downloads\MBB_holdings.csv", skiprows=9)["ISIN"].dropna().iloc[:-6])]["Identifier_UPI"]
upis

# upis = pd.read_csv(r"C:\Users\chris\Downloads\tba_mortgage_option_upis_US01F_US21H.csv")["UPI"]
# upis

3285    QZQW98KWZX1F
3286    QZH5X6THG4ST
3287    QZMN7CZ1P5ZJ
3323    QZGTKFFVVS8Z
3338    QZXW78J2C44S
3498    QZC1G62W6Z1V
3568    QZQMCPKNBQMD
3578    QZSVLQ7SBS0R
3591    QZFN172VB9LF
3596    QZ3BVQT1R704
3694    QZ3KJWCW9BD3
3695    QZW0T05ZW5TM
3696    QZ5KDHTLJGDG
3699    QZ9F9X2RDWJW
3700    QZ2HJR4RVXLG
3701    QZTHQL277K9M
3702    QZ70BMBNMGCB
3910    QZTQNPHC16C8
3913    QZTR83199F8L
3915    QZ4D6M59TDKD
3924    QZZVRVXNRTXS
3928    QZZ39Q6PWK91
4616    QZ677GNMB38M
5567    QZ9VJ8RDTM9Q
5568    QZ4RSG4KPC0S
Name: Identifier_UPI, dtype: object

In [175]:
df[df["Unique Product Identifier"].isin(upis)]

# df[df["Unique Product Identifier"].isin(upis)]["Unique Product Identifier"].value_counts()
# .iloc[0].to_dict()
# with pd.option_context("display.max_rows", 1000):
# 	# display(df[df["UPI Underlier Name"].str.lower().str.contains("ice swap")]["UPI Underlier Name"].value_counts())
# 	display(df[df["UPI Underlier Name"].str.lower().str.contains("ice swap")]["UPI Underlier Name"].value_counts())

# df[df["UPI Underlier Name"] == "USD-SOFR vs USD-SOFR"]["Unique Product Identifier"].value_counts()

# df[(df["Effective Date"].dt.date == datetime.date(2026, 6, 17)) & (df["Expiration Date"].dt.date == datetime.date(2026, 7, 29))].to_csv("june_fomc_dated_sdr_trades.csv")

,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name


In [106]:
# .iloc[1].to_dict()
# .iloc[0].to_dict()

# 0.00955 * 10_000
# .iloc[-1].to_dict()

cms_spread_options_single_look = [
    "QZQHDJR8ZR2C",
	"QZSP0NTLKKFJ",
	"QZP43JLWTM5W",
	"QZVH3T5N6GJ9",
	"QZD8FJ9CGMZ2",
	"QZH64P6BDDFN",
	"QZKC9F9FV3ZV",
]

# df[(df["Unique Product Identifier"].isin(cms_spread_options_single_look))]

df[df["UPI Underlier Name"] == "USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND"].to_csv("USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND sdr trades.csv")

In [ ]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=False, merge_package_legs=False)
sdf

In [7]:
sdf.to_csv("usd_swaps_sdr_classification_data.csv",index=False)

In [108]:
from SDRUtils.anna_dsb_upis.utils import AnnaDSBFetcher

anna_dsb_fetcher = AnnaDSBFetcher()
bearer_token = "eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6ImN4WExKNEZ5UVAxdnl0dEtGX1g5dCJ9.eyJodHRwczovL3Byb2QuYW5uYS1kc2IuY29tL3VzZXJuYW1lIjoidm9ub2Y5MzE1NkBkb2xvZmFuLmNvbSIsImh0dHBzOi8vcHJvZC5hbm5hLWRzYi5jb20vZmRsVDBBY2Nlc3NVUEkiOmZhbHNlLCJodHRwczovL3Byb2QuYW5uYS1kc2IuY29tL3NlYXJjaExpbWl0VVBJIjo1LCJodHRwczovL3Byb2QuYW5uYS1kc2IuY29tL2dyb3VwSWRzIjpbIjIwLjEyMDAuMS9VUElfUmVhZCJdLCJpc3MiOiJodHRwczovL2F1dGguYW5uYS1kc2IuY29tLyIsInN1YiI6ImF1dGgwfDY5OWRhZjBmNWZjY2IyYzZhZmI2ZDY2MCIsImF1ZCI6WyJndWkiLCJodHRwczovL2NmLWFubmEtZHNiLmV1LmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3NzIwNDU5NjcsImV4cCI6MTc3MjA0Njg2Nywic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBlbWFpbCBvZmZsaW5lX2FjY2VzcyIsImF6cCI6Ikg3SEVMdVhBakZMSFJ0NW5aanJhMklZdmZKYzRHb1NFIn0.lRRaTezw_7VWolo4XOpBxvKbWzRCvZB7etTPik4HYJhRYICiB8p3Ie1dXNUm6GpfzwLV_1HGGJDSxfQAP5C1Zh1ssdCjUkSFoi_IPMwZ62hOS7t0pi2SMGLIAwjl_NcbzfQuAEXnUdTQ7cJikcC1gAQzD2I7IhdHJdjwvviB7Twygkum3wKpP6mr3zLllJFzADH_v74Z8aBHEAXML4QyFMGRjMho6YfuH3nCTRgiB7as5W8iasxZ-GOKXnoTnFRn0R2CkB-kH2YT1RjRGXJwVGp2YKYfVfeHZT9TNVUYIDKvdrX7hNUlYwCWUOMzoyD0YnEHbmIgF4-g7LWUcQlrHA" 

temp = anna_dsb_fetcher.get_anna_dsb_upis(asset_class="Other", instrument_type="Option", product="Non_Standard", token=bearer_token, num_of_iterations=1000)
temp

FETCHING ANNA DSB UPI DATASETS...: 100%|██████████| 1000/1000 [01:18<00:00, 12.76it/s]


,TemplateVersion,Header_AssetClass,Header_InstrumentType,Header_UseCase,Header_Level,Identifier_UPI,Identifier_Status,Identifier_StatusReason,Identifier_LastUpdateDateTime,Derived_ClassificationType,...,Attributes_UnderlyingAssetClass_Rates_OtherLegReferenceRateTermUnit,Attributes_UnderlyingAssetClass_Foreign_Exchange_PlaceofSettlement,Attributes_UnderlyingAssetClass_Credit_UnderlyingInstrumentISIN,Attributes_UnderlyingAssetClass_Credit_DebtSeniority,Attributes_UnderlyingAssetClass_Equity_UnderlyingInstrumentIndex,Attributes_UnderlyingAssetClass_Commodities_UnderlyingInstrumentIndexProp,Attributes_UnderlyingAssetClass_Commodities_UnderlyingInstrumentIndex,Attributes_UnderlyingAssetClass_Equity_UnderlyingInstrumentIndexProp,Attributes_UnderlyingAssetClass_Credit_UnderlyingInstrumentLEI,Attributes_UnderlyingAssetClass_Credit_UnderlyingInstrumentIndexProp
0,1,Other,Option,Non_Standard,UPI,QZSRPJTPQNK8,New,,2025-10-21T15:53:26,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Other,Option,Non_Standard,UPI,QZHBC2XQTKBP,New,,2025-10-21T11:03:26,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Other,Option,Non_Standard,UPI,QZGT0R4D558L,New,,2025-10-21T11:01:26,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Other,Option,Non_Standard,UPI,QZFMHMVM6Z62,New,,2025-10-17T04:37:48,HMMDVC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Other,Option,Non_Standard,UPI,QZ6VJ6JBGL9D,New,,2025-10-17T04:16:27,HMMDMC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
574,1,Other,Option,Non_Standard,UPI,QZLVBN9FMQK8,New,,2024-12-02T18:03:49,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
575,1,Other,Option,Non_Standard,UPI,QZC37H7QFDV5,New,,2024-12-02T17:51:26,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
576,1,Other,Option,Non_Standard,UPI,QZG7QR34ZS45,New,,2024-11-27T14:24:16,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
577,1,Other,Option,Non_Standard,UPI,QZJHW6ZZ54QQ,New,,2024-11-22T18:06:35,HMMADC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [109]:
temp.to_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Other-Option-Non_Standard.csv",index=False)